In [ ]:
## Libraries
import numpy as np
import pandas as pd

from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors

## Config
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")   # all dates >= this are test

# <<< FILL THESE FROM ARX CV RESULTS >>>
best_lag_set = [1, 2, 12]
best_params  = {
    "alpha": 1.0,   # example; replace with your chosen alpha
}

# continuous features
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

# one-hot / categorical features
categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

## Metric functions
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result

## Load data
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.sort_values([ENTITY_COL, TIME_COL])

## Evaluation – ARX (Ridge), NO STL

# =========================================================
# LAGGED TARGET FEATURES (best_lag_set) ACROSS FULL PANEL
# =========================================================
required_lag_cols = []
for lag in best_lag_set:
    col = f"{TARGET_COL}_lag{lag}"
    df[col] = df.groupby(ENTITY_COL)[TARGET_COL].shift(lag)
    required_lag_cols.append(col)

# =========================================================
# LA FIXED EFFECTS (ONE-HOT)
# =========================================================
la_dummies = pd.get_dummies(
    df[ENTITY_COL].astype("category"),
    prefix="LA",
    drop_first=True,
)
df = pd.concat([df, la_dummies], axis=1)
la_cols = list(la_dummies.columns)

# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_test  = (df[TIME_COL] >= TEST_START_DATE)

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# Need all lag columns present
df_train = df_train.dropna(subset=required_lag_cols)
df_test  = df_test.dropna(subset=required_lag_cols)

feature_cols = continuous_cols + categorical_cols + la_cols + required_lag_cols
df_train = df_train.dropna(subset=feature_cols)
df_test  = df_test.dropna(subset=feature_cols)

# =========================================================
# STANDARDISE CONTINUOUS + LAG FEATURES
# =========================================================
scale_cols = continuous_cols + required_lag_cols
scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
df_test[scale_cols]  = scaler.transform(df_test[scale_cols])

# =========================================================
# BUILD MATRICES
# =========================================================
feature_cols = continuous_cols + categorical_cols + la_cols + required_lag_cols

X_train = df_train[feature_cols]
y_train = df_train[TARGET_COL].values

X_test  = df_test[feature_cols]
y_test  = df_test[TARGET_COL].values

# =========================================================
# TRAIN FINAL ARX (Ridge)
# =========================================================
model = Ridge(
    alpha=best_params["alpha"],
    fit_intercept=True,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

df_test["y_pred"] = y_pred

# =========================================================
# GLOBAL ACCURACY
# =========================================================
global_mae   = mae(y_test, y_pred)
global_rmse  = rmse(y_test, y_pred)
global_smape = smape(y_test, y_pred)
global_mase  = mase(y_test, y_pred, y_train, m=12)

print("=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test
    .groupby(ENTITY_COL)
    .apply(lambda g: mae(g[TARGET_COL], g["y_pred"]))
)

median_mae = la_mae.median()
p75_mae    = la_mae.quantile(0.75)

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
df_test["resid"] = df_test[TARGET_COL] - df_test["y_pred"]

# Mean residual per LA over test period
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

# Robust centroid extraction (avoids NaNs)
centroids = (
    df_test
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
    .loc[la_resid_mean.index]
)

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"LAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_effective = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_effective,
        eps=1e-8,
        symmetric=True,
        row_standardize=True,
        permutations=999,
        random_state=42,
    )

    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = (
    df_test
    .groupby(TIME_COL)["resid"]
    .mean()
    .sort_index()
)

lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])

print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# OPTIONAL: DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")

# =========================================================
# Result output
# =========================================================
output_path = "../../results/arx_ridge_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "ARX_Ridge",
    "lag_set": str(best_lag_set),
    "params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": I_moran["I"],
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")
